# 시행계획 시도 분리·정제 공통 유틸 사용 템플릿

이 파일을 복사한 뒤 파일명을 `YYYYMMDD_EDA_{연도}_전국_시도분리정제.ipynb` 형식으로 변경한다.

복사 후 반드시 확인할 항목:

1. `YEAR`, `SOURCE_FILE`, `SOURCE_SHEET`
2. 해당 연도의 0원 표기인 `ZERO_BUDGET_TOKENS`
3. 연도 고유 반복 머리글인 `EXTRA_HEADER_PATTERNS` — 2016~2020(제3차 기본계획) 원본은 시도별 단위표기 헤더 행("(단위 : 백만원)" 등)이 있는지 반드시 확인하고, 있으면 `UNIT_NOTATION_PATTERN`을 추가한다 (`src/features/pipeline_common.py` 참고, 이슈 #24)
4. **재원구분 이중계상 여부 `NEEDS_FUNDING_SOURCE_CLEANUP`** — 2016~2019(제3차 기본계획) 원본은 세부사업 하나가 계/국비/지방비(도비·시군비·비예산 등)로 최대 3~4행 중복 표기돼 있다. 원본 `사업분류재정구분` 값이 "계"/"국비"/"지방비" 등인지 확인 후 True로 변경한다 (`drop_exact_duplicate_rows`, `select_total_budget_rows`, 이슈 #26)
5. 소계 QA 허용오차인 `QA_TOLERANCE`
6. LLM 실행 여부와 연도별 체크포인트 경로

> 원본 예산 컬럼은 덮어쓰지 않는다. LLM 셀은 체크포인트와 API 키를 확인하기 전 실행하지 않는다.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.llm_refine import (  # noqa: E402
    call_llm_once,
    clean_sentence,
    needs_llm_rerun,
    numbers_preserved,
)
from src.features.pipeline_common import (  # noqa: E402
    assign_labels,
    build_subtotal_qa,
    calculate_budget_changes,
    classify_row,
    clean_text,
    drop_exact_duplicate_rows,
    get_sido_dir,
    normalize_budget_type,
    select_total_budget_rows,
    to_numeric_budget,
)
from src.features.text_match import (  # noqa: E402
    check_pattern,
    check_pattern_broad,
    dedup_label,
    extract_via_regex,
    find_near_duplicates,
)

print(f"프로젝트 루트: {PROJECT_ROOT}")

## 1. 연도·입력·예외 설정

아래 셀은 복사한 노트북에서 가장 먼저 수정한다. `비예산`, `·` 등을 0으로 처리할지는 해당 연도 원본을 확인한 뒤 결정한다.

In [ ]:
YEAR = 2024  # TODO: 대상 연도로 수정
PREVIOUS_YEAR = YEAR - 1

CURRENT_BUDGET_COL = f"{YEAR}년 예산"
PREVIOUS_BUDGET_COL = f"{PREVIOUS_YEAR}년 예산"
CURRENT_NUM_COL = f"{YEAR}년_예산_num"
PREVIOUS_NUM_COL = f"{PREVIOUS_YEAR}년_예산_num"

DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
REPORTS_DIR = PROJECT_ROOT / "reports"
CONFIG_DIR = PROJECT_ROOT / "configs"

SOURCE_FILE = DATA_DIR / "칼럼정렬" / f"TODO_{YEAR}_칼럼정렬.xlsx"  # TODO
SOURCE_SHEET = "정리본_자동"
TABLE1_FILE = DATA_DIR / f"TODO_{YEAR}_원본.xlsx"  # 원본 대조 시 사용

ZERO_BUDGET_TOKENS = ("-",)  # 예: 2023은 검토 후 ("-", "·", "비예산")
# 예: 2016~2020(제3차 기본계획)은 시도별 "(단위 : 백만원)" 헤더 행이 있는지 원본에서 확인 후
#     EXTRA_HEADER_PATTERNS = (UNIT_NOTATION_PATTERN,) 로 채운다. 2021년 이후 원본은 보통 불필요.
EXTRA_HEADER_PATTERNS = ()
# 2016~2019(제3차 기본계획)는 사업분류재정구분이 계/국비/지방비 등 재원 구분이라 이중계상
# 위험이 있다. True로 바꾸면 classify_row 전에 drop_exact_duplicate_rows +
# select_total_budget_rows를 적용한다 (이슈 #26). 2020년 이후(공통/자체 체계)는 False 유지.
NEEDS_FUNDING_SOURCE_CLEANUP = False
QA_TOLERANCE = 0
RUN_LLM = False

CHECKPOINT_PATH = INTERIM_DIR / f"{YEAR}_llm_정제_체크포인트.csv"
QA_PATH = REPORTS_DIR / f"{YEAR}_전국_QA_검증결과.csv"

## 2. 데이터 로드와 기본 확인

In [ ]:
if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"SOURCE_FILE을 실제 파일로 수정하세요: {SOURCE_FILE}")

df_raw = pd.read_excel(SOURCE_FILE, sheet_name=SOURCE_SHEET)

required_columns = {
    "지역",
    "세부사업명",
    "사업분류재정구분",
    CURRENT_BUDGET_COL,
    PREVIOUS_BUDGET_COL,
    "주요내용",
    "원본행",
}
missing_columns = required_columns.difference(df_raw.columns)
if missing_columns:
    raise KeyError(f"입력 파일에 필요한 컬럼이 없습니다: {sorted(missing_columns)}")

print("데이터 크기:", df_raw.shape)
print("지역 수:", df_raw["지역"].nunique())
display(df_raw.head())

## 3. 행 분류·계층 전파·숫자 변환

소계 QA에 원본 소계 행의 숫자값도 필요하므로 `df_labeled` 전체에 숫자 컬럼을 만든다.

In [ ]:
if NEEDS_FUNDING_SOURCE_CLEANUP:
    df_raw = drop_exact_duplicate_rows(df_raw)
    df_raw = select_total_budget_rows(
        df_raw,
        budget_cols=[CURRENT_BUDGET_COL, PREVIOUS_BUDGET_COL],
        zero_tokens=ZERO_BUDGET_TOKENS,
    )
    print("재원구분(계/국비/지방비) 정리 후 행 수:", len(df_raw))

df_raw["사업행구분"] = df_raw["세부사업명"].apply(
    lambda value: classify_row(value, extra_header_patterns=EXTRA_HEADER_PATTERNS)
)

# TODO: 특정 연도에 예산이 모두 결측인 페이지 머리글이 있다면 여기서 헤더반복으로 보정

df_labeled = df_raw.groupby("지역", group_keys=False).apply(assign_labels)
df_labeled["지역"] = df_raw.loc[df_labeled.index, "지역"]
df_labeled[CURRENT_NUM_COL] = to_numeric_budget(
    df_labeled[CURRENT_BUDGET_COL], zero_tokens=ZERO_BUDGET_TOKENS
)
df_labeled[PREVIOUS_NUM_COL] = to_numeric_budget(
    df_labeled[PREVIOUS_BUDGET_COL], zero_tokens=ZERO_BUDGET_TOKENS
)

df_leaf = df_labeled[df_labeled["사업행구분"] == "세부사업"].copy()
print(df_labeled["사업행구분"].value_counts())
print("세부사업 행 수:", len(df_leaf))

## 4. 텍스트 정제와 예산 재계산

In [ ]:
for column in ["세부사업명", "대분류", "중분류"]:
    df_leaf[column] = clean_text(df_leaf[column])

df_leaf["주요내용"] = clean_text(df_leaf["주요내용"], strip_leading_bullet=True)
df_leaf["사업분류재정구분"] = normalize_budget_type(df_leaf["사업분류재정구분"])

budget_changes = calculate_budget_changes(
    df_leaf[CURRENT_NUM_COL],
    df_leaf[PREVIOUS_NUM_COL],
)
df_leaf[["당해예산", "전년도예산", "증감액", "증감율"]] = budget_changes
display(df_leaf[["지역", "세부사업명", "당해예산", "전년도예산", "증감액", "증감율"]].head())

## 5. 중분류 소계 QA와 저장

QA 계산은 공통 함수가 담당하고, 연도별 파일 저장은 노트북에서 담당한다.

In [ ]:
qa = build_subtotal_qa(
    df_labeled,
    budget_col=CURRENT_NUM_COL,
    tolerance=QA_TOLERANCE,
)

display(qa["결과"].value_counts(dropna=False))
display(qa[qa["결과"] == "불일치"])

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
qa.to_csv(QA_PATH, index=False, encoding="utf-8-sig")
print(f"QA 저장 완료: {QA_PATH}")

## 6. 주요내용 라벨 정리·부분 추출·유사 사업명 후보

In [ ]:
df_leaf["주요내용"] = df_leaf["주요내용"].apply(dedup_label)
df_leaf["주요내용_패턴"] = df_leaf["주요내용"].apply(check_pattern)
df_leaf["주요내용_패턴_확장"] = df_leaf["주요내용"].apply(check_pattern_broad)

regex_extracted = pd.DataFrame(
    df_leaf["주요내용"].apply(extract_via_regex).tolist(),
    index=df_leaf.index,
)
df_leaf["지원대상"] = regex_extracted["지원대상"]
df_leaf["지원내용_상세"] = regex_extracted["지원내용"]

near_duplicates = find_near_duplicates(df_leaf, threshold=90)
print("유사 사업명 검토 후보:", len(near_duplicates))
display(near_duplicates.head(20))

## 7. 원본 Table 1 대조(필요 시)

원본 파일의 컬럼 배치가 다르면 `column_indices`를 해당 연도에 맞게 변경한다.

In [ ]:
# 예시: 실제 행 번호로 교체한 뒤 주석을 해제한다.
# df_table1 = pd.read_excel(TABLE1_FILE, sheet_name="Table 1", header=None)
# display(show_table1_around(df_table1, center_excel_row=100, window=3, label="원본 대조"))

## 8. LLM 보존형 교정(선택)

`RUN_LLM=True`로 바꾸기 전에 `.env`, 설정 파일, 연도별 체크포인트 파일명을 확인한다. 아래 셀은 연결 예시이며, 대량 실행 시 기존 노트북의 청크 단위 체크포인트 저장 로직을 함께 사용한다.

In [ ]:
if RUN_LLM:
    import os
    from functools import partial

    import yaml
    from dotenv import load_dotenv
    from openai import OpenAI

    load_dotenv(PROJECT_ROOT / ".env")
    api_key = os.getenv("UPSTAGE_API_KEY")
    if not api_key:
        raise RuntimeError("UPSTAGE_API_KEY가 없습니다.")

    with (CONFIG_DIR / "llm_extraction.yaml").open(encoding="utf-8") as file:
        llm_cfg = yaml.safe_load(file)

    client = OpenAI(api_key=api_key, base_url=llm_cfg["upstage"]["base_url"])
    call_once = partial(call_llm_once, client=client, llm_config=llm_cfg)

    # 안전 확인용 단건 예시. 대량 처리는 체크포인트 로직을 붙인 뒤 수행한다.
    sample = df_leaf[df_leaf["주요내용"].notna()].iloc[0]
    sample_cleaned = clean_sentence(sample["세부사업명"], sample["주요내용"], call_once=call_once)
    print("원문:", sample["주요내용"])
    print("정제:", sample_cleaned)
    print("숫자보존:", numbers_preserved(sample["주요내용"], sample_cleaned))
    print("재실행대상:", needs_llm_rerun(sample_cleaned))
else:
    print("RUN_LLM=False: LLM 호출을 건너뜁니다.")

## 9. 시도별 최종 저장

최종 컬럼은 해당 연도 작업에서 확정한 스키마로 수정한다. 같은 파일을 중간과 최종 단계에서 중복 저장하지 않는다.

In [ ]:
output_cols = [
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "당해예산",
    "전년도예산",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
]
missing_output_cols = set(output_cols).difference(df_leaf.columns)
if missing_output_cols:
    raise KeyError(f"최종 저장 전에 필요한 컬럼을 확인하세요: {sorted(missing_output_cols)}")

for sido, group in df_leaf.groupby("지역"):
    sido_dir = get_sido_dir(INTERIM_DIR, sido)
    output_path = sido_dir / f"{YEAR}_{sido}_세부사업_정제.csv"
    group[output_cols].to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"{sido}: {len(group)}행 저장 -> {output_path}")

## 완료 전 체크리스트

- [ ] 17개 시도 포함 여부 확인
- [ ] 행 유형 분포 확인
- [ ] 연도 고유 헤더·연속행 후보 원본 대조
- [ ] 재원구분(계/국비/지방비) 이중계상 여부 확인 및 `NEEDS_FUNDING_SOURCE_CLEANUP` 반영 (2016~2019)
- [ ] 중분류 소계 QA 불일치 원인 기록
- [ ] 증감액·증감율 재계산 결과 확인
- [ ] LLM 숫자 보존 불일치 원본 대조
- [ ] wide/long 행 수와 컬럼 스키마 확인
- [ ] 최종 CSV `utf-8-sig` 저장 확인
- [ ] 커널 재시작 후 전체 실행
- [ ] 리팩터링 전후 QA 수치 비교